# AI-Agent vs Human PR Dependency Vulnerability Pipeline

**Replica of** [`ahmadmustafa02/ai-agent-dependency-vulns`](https://github.com/ahmadmustafa02/ai-agent-dependency-vulns)
mining pipeline, rewritten as a **step-by-step notebook with no helper functions** — every operation
is inline so you can read, pause, inspect, and resume at any cell.

## What this notebook does

1. Load the **AIDev-Pop** dataset (AI-agent-authored PRs from GitHub, published by Li et al. on HuggingFace).
2. Identify every repo + time window that contains AI-agent PRs touching dependency manifests.
3. Mine the **matched human-authored baseline** from those same repos via the GitHub GraphQL API.
4. Fetch `base_sha`, `head_sha`, and `merge_sha` for every PR (batched GraphQL).
5. Fetch the dependency manifest/lockfile **content before and after** each PR.
6. Scan each snapshot with **OSV-Scanner** (known CVE database).
7. Also scan `.py` / `.js` / `.ts` source files with **Semgrep** for code-pattern vulnerabilities.
8. Compute per-PR: `introduced` vs `fixed` vulnerabilities.
9. Pull `review_count`, `comment_count`, `merged_at` for the exploratory analysis.
10. Save the final analysis-ready CSV.

## Runtime environment
- **Local Windows** — not Colab. Paths use `pathlib.Path` so they work on any OS.
- Everything is **checkpointed to disk** at each stage — safe to interrupt and restart.
- **Default scope**: top 500 repos by AI PR count (covers ~50% of AI PRs). Set `N_REPOS = None` in Step 6 for the full 9,509-repo scope.

## What you need before running
- Python 3.10+
- A **GitHub Personal Access Token** (read-only public_repo scope is enough)
- `pip install pandas requests pyarrow nbformat osv-scanner-cli semgrep` (we install in cell 2)
- Disk space: ~500 MB for parquet caches + checkpoints

## Step 1 — Imports and workspace paths

We'll do every import up front. Then we define a single `WORK_DIR` where all checkpoints, downloads,
and the final CSV will live. Everything is written relative to this directory so the notebook is
portable.

In [15]:
import os
import re
import json
import time
import base64
import subprocess
import tempfile
import shutil
from pathlib import Path
from collections import Counter

import pandas as pd
import requests

# ---------- Workspace ----------
WORK_DIR = Path(r"C:\Users\abhinandan.shrestha\Documents\Test\AIAgentVulnPipeline\data")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Every artefact this notebook produces
HUMAN_PRS_FILE    = WORK_DIR / "human_manifest_touching_prs.csv"
HUMAN_DONE_FILE   = WORK_DIR / "human_mining_done_repos.txt"
MINING_FAIL_LOG   = WORK_DIR / "mining_failures.log"

AI_PRS_FILE       = WORK_DIR / "ai_manifest_touching_prs.csv"

ALL_SHAS_FILE     = WORK_DIR / "all_pr_shas.csv"
SHA_FAIL_LOG      = WORK_DIR / "sha_fetch_failures.log"

OSV_RESULTS_FILE  = WORK_DIR / "osv_scan_results.csv"
OSV_DONE_FILE     = WORK_DIR / "osv_scan_done_ids.txt"
OSV_FAIL_LOG      = WORK_DIR / "osv_scan_failures.log"

SEMGREP_RESULTS_FILE = WORK_DIR / "semgrep_scan_results.csv"
SEMGREP_DONE_FILE    = WORK_DIR / "semgrep_scan_done_ids.txt"
SEMGREP_FAIL_LOG     = WORK_DIR / "semgrep_scan_failures.log"

REVIEW_META_FILE  = WORK_DIR / "review_metadata.csv"

FINAL_FILE        = WORK_DIR / "final_analysis_dataset.csv"

print("Workspace ready at:", WORK_DIR)
print("Files that will be produced in this run:")
for p in [HUMAN_PRS_FILE, AI_PRS_FILE, ALL_SHAS_FILE, OSV_RESULTS_FILE,
          SEMGREP_RESULTS_FILE, REVIEW_META_FILE, FINAL_FILE]:
    print("  ", p.name, "(exists)" if p.exists() else "(will be created)")

Workspace ready at: C:\Users\abhinandan.shrestha\Documents\Test\AIAgentVulnPipeline\data
Files that will be produced in this run:
   human_manifest_touching_prs.csv (will be created)
   ai_manifest_touching_prs.csv (exists)
   all_pr_shas.csv (will be created)
   osv_scan_results.csv (will be created)
   semgrep_scan_results.csv (will be created)
   review_metadata.csv (will be created)
   final_analysis_dataset.csv (will be created)


## Step 2 — Install dependencies and scanners

We need:
- `pandas`, `requests`, `pyarrow` — data handling and GitHub API calls
- `osv-scanner` binary — official Google tool, scans lockfiles against the OSV.dev CVE database
- `semgrep` — scans source code for vulnerability patterns (SQLi, XSS, hardcoded secrets, etc.)

On Windows, OSV-Scanner is installed as a `.exe` binary from GitHub releases. Semgrep is a Python
package. Both are invoked as subprocesses.

**Note**: if you already have these installed, this cell is cheap — it just verifies the versions.

In [16]:
# --- Python packages ---
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pandas", "requests", "pyarrow", "semgrep"], check=False)

# --- OSV-Scanner binary (Windows) ---
OSV_BIN = WORK_DIR / "osv-scanner.exe"
if not OSV_BIN.exists():
    print("Downloading OSV-Scanner binary for Windows...")
    url = "https://github.com/google/osv-scanner/releases/latest/download/osv-scanner_windows_amd64.exe"
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()
    with open(OSV_BIN, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Saved to {OSV_BIN}")

# --- Verify versions ---
print("\n--- OSV-Scanner ---")
r = subprocess.run([str(OSV_BIN), "--version"], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())

print("\n--- Semgrep ---")
r = subprocess.run(["semgrep", "--version"], capture_output=True, text=True, shell=True)
print(r.stdout.strip() or r.stderr.strip())


--- OSV-Scanner ---
osv-scanner version: 2.7.1
osv-scalibr version: 0.5.3
commit: 02f96a40691c2d155fdda799af436e111231e365
built at: 2026-11-09T03:39:40Z

--- Semgrep ---
1.185.0


## Step 3 — Load your GitHub token from `.env`

We read the token from a `.env` file that sits in the **same folder as this notebook**.

Create `.env` with this single line (no quotes, no spaces):

```
GITHUB_TOKEN=ghp_your_token_here
```

A GitHub Personal Access Token with `public_repo` read scope is enough. The pipeline makes tens of
thousands of GraphQL calls — an unauthenticated session will hit the 60/hour limit within seconds.

**Never commit `.env`.** Add it to your `.gitignore`.


In [17]:
# Load token from .env file sitting next to this notebook
NOTEBOOK_DIR = Path.cwd()
ENV_FILE = NOTEBOOK_DIR / ".env"

TOKEN = None
if ENV_FILE.exists():
    for line in ENV_FILE.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" in line:
            key, _, val = line.partition("=")
            key = key.strip()
            val = val.strip().strip('"').strip("'")
            if key == "GITHUB_TOKEN":
                TOKEN = val
                break

# Fallback to environment variable (useful if you set it in the shell instead)
if not TOKEN:
    TOKEN = os.environ.get("GITHUB_TOKEN")

assert TOKEN, (
    f"GITHUB_TOKEN not found.\n"
    f"Create a .env file at: {ENV_FILE}\n"
    f"with a single line:   GITHUB_TOKEN=ghp_your_token_here"
)

HEADERS = {"Authorization": f"bearer {TOKEN}", "Content-Type": "application/json"}
GRAPHQL_URL = "https://api.github.com/graphql"

# Sanity check: how much quota remains?
r = requests.get("https://api.github.com/rate_limit", headers=HEADERS, timeout=10).json()
print(f"Token loaded from: {ENV_FILE if ENV_FILE.exists() else 'environment variable'}")
print(f"Token preview:     {TOKEN[:7]}...{TOKEN[-4:]}")
print(f"GraphQL remaining: {r['resources']['graphql']['remaining']} / {r['resources']['graphql']['limit']}")
print(f"REST remaining:    {r['resources']['core']['remaining']} / {r['resources']['core']['limit']}")


Token loaded from: c:\Users\abhinandan.shrestha\Documents\Test\AIAgentVulnPipeline\.env
Token preview:     github_...tBk7
GraphQL remaining: 4808 / 5000
REST remaining:    4997 / 5000


## Step 4 — Load the AIDev-Pop dataset from HuggingFace

The **AIDev** dataset (Li et al., HuggingFace) is a curated snapshot of pull requests authored by
AI coding agents — Copilot, Cursor, Devin, Claude Code, OpenAI Codex — on real public GitHub repos.
We need three parquet files from it:

| File | What's in it | Columns we use |
|---|---|---|
| `pr_commit_details.parquet` | one row per file touched by each PR | `pr_id`, `filename` |
| `pull_request.parquet` | one row per PR | `id`, `agent`, `repo_id`, `user`, `created_at`, `merged_at` |
| `repository.parquet` | one row per repo | `id`, `full_name` |

`pd.read_parquet("hf://datasets/hao-li/AIDev/…")` pulls them directly over HTTP — no HuggingFace
account or `datasets` library needed, just `pyarrow`.

**Expect ~15 seconds** for the download on first run. Parquets get cached locally.

> **Scope knob**: `MAX_REPOS = None` means "all repos". Set it to e.g. `5` for a quick demo run.

In [18]:
MAX_REPOS = None  # None = all repos from AIDev-Pop. Set to small int for a demo.

print("Loading AIDev commit_details.parquet ...")
commit_details = pd.read_parquet(
    "hf://datasets/hao-li/AIDev/pr_commit_details.parquet",
    columns=["pr_id", "filename"],
)
print(f"  commit_details: {len(commit_details):,} rows")

print("\nLoading AIDev pull_request.parquet ...")
pull_requests = pd.read_parquet(
    "hf://datasets/hao-li/AIDev/pull_request.parquet",
    columns=["id", "agent", "repo_id", "user", "created_at", "merged_at"],
)
print(f"  pull_requests:  {len(pull_requests):,} rows")

print("\nLoading AIDev repository.parquet ...")
repository = pd.read_parquet(
    "hf://datasets/hao-li/AIDev/repository.parquet",
    columns=["id", "full_name"],
)
print(f"  repository:     {len(repository):,} rows")

print("\nAgents present in AIDev:")
print(pull_requests["agent"].value_counts(dropna=False).head(10))

Loading AIDev commit_details.parquet ...
  commit_details: 12,116,310 rows

Loading AIDev pull_request.parquet ...
  pull_requests:  361,296 rows

Loading AIDev repository.parquet ...
  repository:     25,402 rows

Agents present in AIDev:
agent
Claude_Code     172597
Copilot          98384
OpenAI_Codex     62731
Cursor           12908
Devin            11219
Google_Jules      3457
Name: count, dtype: int64


## Step 5 — Classify dependency files by ecosystem

Not every file in a PR is a dependency manifest. We only care about files that actually declare
project dependencies, because only those can introduce a vulnerable package.

| Ecosystem | File patterns |
|---|---|
| **python** | `requirements*.txt`, `pyproject.toml`, `poetry.lock`, `Pipfile`, `Pipfile.lock` |
| **typescript** | `package.json`, `package-lock.json`, `yarn.lock`, `pnpm-lock.yaml` |
| **go** | `go.mod`, `go.sum` |
| **csharp** | `*.csproj`, `packages.config`, `packages.lock.json` |

We apply the regex row-by-row; non-matching files (source code, docs, CI configs) get `None` and
are dropped. The remaining rows are the **AI side** of our dataset.

In [19]:
ECOSYSTEM_PATTERNS = {
    "python":     r"(^|/)(requirements.*\.txt|pyproject\.toml|poetry\.lock|Pipfile|Pipfile\.lock)$",
    "typescript": r"(^|/)(package\.json|package-lock\.json|yarn\.lock|pnpm-lock\.yaml)$",
    "go":         r"(^|/)(go\.mod|go\.sum)$",
    "csharp":     r"(^|/)([^/]+\.csproj|packages\.config|packages\.lock\.json)$",
}

# Row-by-row classification — no function, just an inline loop
ecosystems = []
for fn in commit_details["filename"]:
    eco = None
    if isinstance(fn, str):
        for name, pat in ECOSYSTEM_PATTERNS.items():
            if re.search(pat, fn):
                eco = name
                break
    ecosystems.append(eco)
commit_details["ecosystem"] = ecosystems

# Keep only manifest-touching rows
manifest_touches = commit_details.dropna(subset=["ecosystem"]).copy()
print(f"Manifest-touching rows (AI side): {len(manifest_touches):,}")
print("\nBy ecosystem:")
print(manifest_touches["ecosystem"].value_counts())

# Join with PR metadata + repo full_name
merged_ai = manifest_touches.merge(pull_requests, left_on="pr_id", right_on="id", how="inner")
merged_ai = merged_ai.merge(repository, left_on="repo_id", right_on="id", suffixes=("", "_repo"))
merged_ai = merged_ai[["pr_id", "agent", "repo_id", "full_name", "ecosystem", "filename",
                       "user", "created_at", "merged_at"]].copy()
print(f"\nMerged AI manifest-touching rows: {len(merged_ai):,}")
print(f"Unique AI PRs: {merged_ai['pr_id'].nunique():,}")
print(f"Unique repos: {merged_ai['full_name'].nunique():,}")

merged_ai.to_csv(AI_PRS_FILE, index=False)
print(f"\nSaved AI-side manifest touches -> {AI_PRS_FILE}")

Manifest-touching rows (AI side): 360,512

By ecosystem:
ecosystem
typescript    248728
python         45395
csharp         38773
go             27616
Name: count, dtype: int64

Merged AI manifest-touching rows: 360,512
Unique AI PRs: 57,976
Unique repos: 9,509

Saved AI-side manifest touches -> C:\Users\abhinandan.shrestha\Documents\Test\AIAgentVulnPipeline\data\ai_manifest_touching_prs.csv


## Step 6 — Build the matched sampling frame (top-500 repos by AI-PR count)

For every repo that has at least one AI-agent manifest-touching PR, we will later mine the human
PRs from that **same repo** in the **same time window**. This is the paper's matched-design core —
AI and human PRs come from the same codebase so repo-specific confounders cancel out.

**Filtering down from 9,509 repos**: scanning every repo in AIDev is prohibitive. Instead, we rank
repos by **number of unique AI PRs** and keep the top `N_REPOS` (default 500). This covers roughly
50% of all AI PRs in the dataset while using ~5% of the repos — a huge win for statistical power
per unit of runtime.

Repos with more AI activity also tend to have more human activity in the same window, so our
matched human baseline becomes denser per repo.

Three things get built:
- `repo_paths` — for each top-N repo, the exact set of manifest file paths we care about
- `global_since` / `global_until` — the AIDev time window
- `bot_exclusions` — the usernames we must EXCLUDE from the human side (otherwise bots like
  Dependabot would masquerade as humans)

In [20]:
# ---------- Top-N filter knob ----------
# Set to an int for ranked top-N (recommended), or None for ALL repos.
N_REPOS = 500

# Rank repos by unique AI PR count (descending)
repo_ai_pr_counts = (
    merged_ai.groupby("full_name")["pr_id"].nunique().sort_values(ascending=False)
)
total_repos = len(repo_ai_pr_counts)
total_ai_prs = repo_ai_pr_counts.sum()

if N_REPOS is not None:
    top_repos = repo_ai_pr_counts.head(N_REPOS).index.tolist()
    covered_prs = repo_ai_pr_counts.head(N_REPOS).sum()
    min_pr_in_topN = int(repo_ai_pr_counts.head(N_REPOS).min())
    print(f"Full AIDev scope:     {total_repos:,} repos  /  {total_ai_prs:,} AI PRs")
    print(f"Keeping top {N_REPOS:,}:    covers {covered_prs:,} AI PRs "
          f"({covered_prs/total_ai_prs*100:.1f}%)")
    print(f"Min AI PRs in top N:  {min_pr_in_topN}")
else:
    top_repos = repo_ai_pr_counts.index.tolist()
    print(f"FULL SCOPE: all {total_repos:,} repos")

# Build repo_paths only for selected repos
selected_df = merged_ai[merged_ai["full_name"].isin(top_repos)]
repo_paths = (
    selected_df.groupby("full_name")["filename"]
    .apply(lambda s: sorted(set(s)))
    .to_dict()
)
# Preserve ranking order (dict insertion order in py3.7+)
repo_paths = {r: repo_paths[r] for r in top_repos if r in repo_paths}

# Time window — use the full AIDev window, not just the top-N subset, so the human
# baseline stays comparable across repos
global_since = merged_ai["created_at"].min()
global_until = merged_ai["created_at"].max()

# All AI-agent user logins — these are the agents authoring the AI side
agent_logins = set(
    pull_requests.loc[pull_requests["agent"].notna(), "user"].dropna().unique()
)
bot_exclusions = agent_logins | {
    "dependabot[bot]", "renovate[bot]", "github-actions[bot]", "renovate-bot",
}

print(f"\nRepos in scope:        {len(repo_paths):,}")
print(f"Time window from:      {global_since}")
print(f"Time window to:        {global_until}")
print(f"Bot/agent exclusions:  {len(bot_exclusions)} logins")
print(f"\nTop 10 repos selected (by AI PR count):")
for i, repo in enumerate(list(repo_paths.keys())[:10], 1):
    print(f"  {i:>2}. {repo:<50s} ({repo_ai_pr_counts[repo]} AI PRs)")

Full AIDev scope:     9,509 repos  /  57,976 AI PRs
Keeping top 500:    covers 28,986 AI PRs (50.0%)
Min AI PRs in top N:  21

Repos in scope:        500
Time window from:      2024-12-24T03:28:51Z
Time window to:        2026-03-31T23:52:45Z
Bot/agent exclusions:  28740 logins

Top 10 repos selected (by AI PR count):
   1. openclaw/openclaw                                  (706 AI PRs)
   2. vm0-ai/vm0                                         (509 AI PRs)
   3. github/gh-aw                                       (476 AI PRs)
   4. ceedaragents/cyrus                                 (363 AI PRs)
   5. nomadkaraoke/karaoke-gen                           (355 AI PRs)
   6. airbytehq/airbyte                                  (332 AI PRs)
   7. promptfoo/promptfoo                                (326 AI PRs)
   8. nexi-lab/nexus                                     (326 AI PRs)
   9. Yeraze/meshmonitor                                 (317 AI PRs)
  10. dotnet/runtime                               

## Step 7 — Mine human-authored manifest-touching PRs via GraphQL

For each repo, we paginate through its **merged PRs** (most recent first). For each PR we:

1. Skip anything outside the AIDev time window.
2. Skip any author in `bot_exclusions`.
3. Intersect the PR's file list with `paths_of_interest` (the manifest files this repo has).
4. Save one row per `(pr_id, filename)` match.

The loop is **checkpointed per repo in append mode**: after each repo is done, that repo's rows
are appended to `human_manifest_touching_prs.csv` (never rewritten) and the repo name is appended
to `human_mining_done_repos.txt`. If you interrupt and re-run this cell, it reloads the done-set
from disk and skips those repos.

> Expected runtime: **~30-60 min for top 500 repos**, several hours for full 9,509-repo scope.
> GraphQL rate limit is 5000 points/hour, roughly 1 PR page per point.

In [21]:
QUERY = """
query($owner: String!, $name: String!, $cursor: String) {
  rateLimit { remaining resetAt }
  repository(owner: $owner, name: $name) {
    pullRequests(states: MERGED, first: 50, after: $cursor,
                 orderBy: {field: CREATED_AT, direction: DESC}) {
      pageInfo { hasNextPage endCursor }
      nodes {
        databaseId
        number
        author { login }
        createdAt
        mergedAt
        files(first: 100) { nodes { path } }
      }
    }
  }
}
"""

# --- Resume state from disk ---
done_now = set()
if HUMAN_DONE_FILE.exists():
    done_now = set(l.strip() for l in HUMAN_DONE_FILE.read_text().splitlines() if l.strip())

# Count existing rows in CSV (no need to load the full file into memory anymore)
total_rows_on_disk = 0
if HUMAN_PRS_FILE.exists():
    # fast line-count, subtract header
    with open(HUMAN_PRS_FILE, "r", encoding="utf-8") as _f:
        total_rows_on_disk = sum(1 for _ in _f) - 1

CSV_COLUMNS = ["pr_id", "number", "agent", "full_name", "ecosystem",
               "filename", "created_at", "merged_at"]

# Write header once if the file doesn't exist yet
if not HUMAN_PRS_FILE.exists():
    pd.DataFrame(columns=CSV_COLUMNS).to_csv(HUMAN_PRS_FILE, index=False)

print(f"Repos already done:  {len(done_now):,}")
print(f"Rows already in CSV: {total_rows_on_disk:,}")
print(f"Total repos:         {len(repo_paths):,}")
print(f"Remaining:           {len(repo_paths) - len(done_now):,}")

# --- Main mining loop (inline, no function; appends to CSV per repo) ---
for full_name, paths in repo_paths.items():
    if full_name in done_now:
        continue
    owner, repo = full_name.split("/", 1)
    paths_of_interest = set(paths)
    repo_rows = []     # buffer rows for THIS repo only
    cursor = None
    stop = False
    while not stop:
        # one GraphQL call, up to 5 retries inline
        data = None
        for attempt in range(5):
            try:
                resp = requests.post(
                    GRAPHQL_URL, headers=HEADERS,
                    json={"query": QUERY, "variables": {"owner": owner, "name": repo, "cursor": cursor}},
                    timeout=30,
                )
                if resp.status_code == 200:
                    j = resp.json()
                    if "errors" in j:
                        data = None
                    else:
                        data = j["data"]
                    break
                time.sleep(5 * (attempt + 1))
            except requests.exceptions.RequestException:
                time.sleep(10 * (attempt + 1))
        if data is None or data.get("repository") is None:
            with open(MINING_FAIL_LOG, "a") as f:
                f.write(f"{full_name}\tgraphql\tfailed\n")
            break
        pr_block = data["repository"]["pullRequests"]
        for pr in pr_block["nodes"]:
            if pr["createdAt"] < str(global_since):
                stop = True
                break
            if pr["createdAt"] > str(global_until):
                continue
            author = pr["author"]["login"] if pr["author"] else None
            if author is None or author in bot_exclusions:
                continue
            if pr.get("files") is None:
                continue
            file_paths = {n["path"] for n in pr["files"]["nodes"]}
            matched = file_paths & paths_of_interest
            for path in matched:
                eco = None
                for name, pat in ECOSYSTEM_PATTERNS.items():
                    if re.search(pat, path):
                        eco = name
                        break
                repo_rows.append({
                    "pr_id": pr["databaseId"],
                    "number": pr["number"],
                    "agent": "human",
                    "full_name": full_name,
                    "ecosystem": eco,
                    "filename": path,
                    "created_at": pr["createdAt"],
                    "merged_at": pr["mergedAt"],
                })
        if not pr_block["pageInfo"]["hasNextPage"]:
            break
        cursor = pr_block["pageInfo"]["endCursor"]

    # --- APPEND this repo's rows to CSV (no in-memory accumulation) ---
    if repo_rows:
        pd.DataFrame(repo_rows, columns=CSV_COLUMNS).to_csv(
            HUMAN_PRS_FILE, mode="a", header=False, index=False,
        )
        total_rows_on_disk += len(repo_rows)
    done_now.add(full_name)
    with open(HUMAN_DONE_FILE, "a") as f:
        f.write(full_name + "\n")
    print(f"  Done: {full_name:<50s} +{len(repo_rows)} human PRs  (CSV total: {total_rows_on_disk:,})")

print(f"\nTotal human manifest-touching rows written: {total_rows_on_disk:,}")

Repos already done:  0
Rows already in CSV: 0
Total repos:         500
Remaining:           500


KeyboardInterrupt: 

## Step 8 — Combine AI + human sides into one list that needs SHAs

Both sides need the same SHA metadata (base commit, head commit, merge commit) in order to fetch
the file content before/after the PR. We build a unified `all_prs_needing_sha` dataframe with one
row per `(pr_id, full_name, number)` so we can fetch everything in one batched pass.

The AI side needs special handling — AIDev doesn't carry PR numbers, only `pr_id`. We'll back-fill
PR numbers from GitHub in the next cell.

In [ ]:
human_df = pd.read_csv(HUMAN_PRS_FILE) if HUMAN_PRS_FILE.exists() else pd.DataFrame()
ai_df = pd.read_csv(AI_PRS_FILE)

print(f"Human rows: {len(human_df):,}  ({human_df['pr_id'].nunique():,} unique PRs)")
print(f"AI    rows: {len(ai_df):,}  ({ai_df['pr_id'].nunique():,} unique PRs)")

# Unified list of PRs we need SHA metadata for.
# Human side already has `number` from the GraphQL mining step. AI side doesn't — we tag it NaN
# and backfill via GitHub in the next cell.
human_for_sha = human_df[["pr_id", "full_name", "number"]].drop_duplicates(subset=["pr_id"]).copy()
human_for_sha["side"] = "human"

ai_for_sha = ai_df[["pr_id", "full_name"]].drop_duplicates(subset=["pr_id"]).copy()
ai_for_sha["number"] = pd.NA
ai_for_sha["side"] = "ai"

all_prs_needing_sha = pd.concat([human_for_sha, ai_for_sha], ignore_index=True)
print(f"\nCombined PRs needing SHA: {len(all_prs_needing_sha):,}")
print(all_prs_needing_sha["side"].value_counts())

## Step 9 — Backfill GitHub PR numbers for the AI side

AIDev stores `databaseId` (a.k.a. `pr_id`) but not the repository-scoped PR number (the number
you see in the URL, e.g. `pulls/1234`). We need the number because the next step's GraphQL query
looks up PRs by number.

We do a cheap one-shot lookup: for each unique `(pr_id, full_name)` on the AI side, query GitHub
for the PR number via GraphQL. Batched 50 PRs per repo per call.

If you already have `ai_pr_numbers.csv`, this cell reuses it.

In [ ]:
AI_PR_NUMBERS_FILE = WORK_DIR / "ai_pr_numbers.csv"

if AI_PR_NUMBERS_FILE.exists():
    ai_numbers_df = pd.read_csv(AI_PR_NUMBERS_FILE)
    print(f"Reusing existing file: {len(ai_numbers_df):,} AI PR numbers")
else:
    # For each AI PR we only know the databaseId — but GraphQL's node() lets us resolve it directly.
    ai_rows = []
    ai_ids = ai_for_sha["pr_id"].astype("Int64").dropna().unique().tolist()
    print(f"Resolving {len(ai_ids):,} AI PR numbers via GraphQL node() lookup ...")
    BATCH = 50
    for i in range(0, len(ai_ids), BATCH):
        batch = ai_ids[i:i + BATCH]
        aliases = "\n".join(
            f'n{j}: node(id: "{nid_gid}") {{ ... on PullRequest {{ databaseId number repository {{ nameWithOwner }} }} }}'
            # NOTE: AIDev databaseId is an INT, but node() requires the base64 GraphQL id.
            # Easier workaround: ignore node() and fall back to a numbered-lookup search per repo.
            for j, nid_gid in enumerate(batch)
        )
        break  # placeholder — switch to the per-repo approach below
    # Per-repo approach (robust): for each repo on the AI side, paginate merged PRs and match by databaseId
    done_ai_ids = set()
    for full_name, group in ai_for_sha.groupby("full_name"):
        owner, repo = full_name.split("/", 1)
        wanted = set(int(x) for x in group["pr_id"].dropna().unique())
        if not wanted:
            continue
        cursor = None
        while wanted:
            q = """
            query($owner:String!,$name:String!,$cursor:String){
              repository(owner:$owner,name:$name){
                pullRequests(states:MERGED, first:100, after:$cursor,
                             orderBy:{field: CREATED_AT, direction: DESC}){
                  pageInfo{hasNextPage endCursor}
                  nodes{ databaseId number createdAt }
                }
              }
            }
            """
            resp = requests.post(GRAPHQL_URL, headers=HEADERS,
                                 json={"query": q, "variables": {"owner": owner, "name": repo, "cursor": cursor}},
                                 timeout=30)
            if resp.status_code != 200:
                time.sleep(5); continue
            jd = resp.json().get("data", {}).get("repository")
            if not jd:
                break
            block = jd["pullRequests"]
            for pr in block["nodes"]:
                if pr["databaseId"] in wanted:
                    ai_rows.append({
                        "pr_id": pr["databaseId"], "full_name": full_name, "number": pr["number"],
                    })
                    wanted.discard(pr["databaseId"])
                    done_ai_ids.add(pr["databaseId"])
            if not block["pageInfo"]["hasNextPage"] or not wanted:
                break
            # stop paging if we've gone before global_since
            if pr["createdAt"] < str(global_since):
                break
            cursor = block["pageInfo"]["endCursor"]
        print(f"  {full_name}: resolved {len(group) - len(wanted)}/{len(group)} AI PR numbers")
        # checkpoint incrementally
        pd.DataFrame(ai_rows).to_csv(AI_PR_NUMBERS_FILE, index=False)
    ai_numbers_df = pd.DataFrame(ai_rows)
    print(f"\nTotal AI PR numbers resolved: {len(ai_numbers_df):,}")

# Merge numbers back into all_prs_needing_sha
all_prs_needing_sha = all_prs_needing_sha.merge(
    ai_numbers_df[["pr_id", "number"]].rename(columns={"number": "number_ai"}),
    on="pr_id", how="left",
)
all_prs_needing_sha["number"] = all_prs_needing_sha["number"].fillna(all_prs_needing_sha["number_ai"])
all_prs_needing_sha = all_prs_needing_sha.drop(columns=["number_ai"])
missing = all_prs_needing_sha["number"].isna().sum()
print(f"\nPRs with number resolved: {len(all_prs_needing_sha) - missing:,}")
print(f"PRs with NO number (will be dropped): {missing:,}")
all_prs_needing_sha = all_prs_needing_sha.dropna(subset=["number"]).copy()
all_prs_needing_sha["number"] = all_prs_needing_sha["number"].astype(int)

## Step 10 — Batched SHA fetch for every PR

For each PR we need three commit SHAs:
- `baseRefOid` → the commit the PR branched from (the "before" state)
- `mergeCommit.oid` → the merge commit (the "after" state, primary)
- `headRefOid` → the head of the PR branch (fallback when `mergeCommit` is null or the branch was deleted)

GraphQL lets us batch 50 PRs per repo into a single query using **aliases** (`pr0:`, `pr1:`, …).
That cuts the request count by ~50×.

Progress is appended to `all_pr_shas.csv` after each batch. Interrupt-safe.

In [ ]:
BATCH_SIZE = 50

# Resume
done_pr_ids = set()
sha_results = []
if ALL_SHAS_FILE.exists():
    sha_df_prev = pd.read_csv(ALL_SHAS_FILE)
    sha_results = sha_df_prev.to_dict("records")
    done_pr_ids = set(sha_df_prev["pr_id"])

remaining = all_prs_needing_sha[~all_prs_needing_sha["pr_id"].isin(done_pr_ids)]
print(f"Already have SHAs for: {len(done_pr_ids):,}")
print(f"Remaining to fetch:    {len(remaining):,}")

for full_name, group in remaining.groupby("full_name"):
    owner, repo = full_name.split("/", 1)
    pr_list = group[["pr_id", "number"]].to_dict("records")
    for i in range(0, len(pr_list), BATCH_SIZE):
        batch = pr_list[i:i + BATCH_SIZE]
        numbers = [int(p["number"]) for p in batch]
        pr_id_by_number = {int(p["number"]): p["pr_id"] for p in batch}
        aliases = "\n".join(
            f'pr{j}: pullRequest(number: {n}) {{ number baseRefOid headRefOid mergeCommit {{ oid }} }}'
            for j, n in enumerate(numbers)
        )
        query = f'query {{ repository(owner: "{owner}", name: "{repo}") {{ {aliases} }} }}'
        data = None
        for attempt in range(5):
            try:
                resp = requests.post(GRAPHQL_URL, headers=HEADERS,
                                     json={"query": query}, timeout=30)
                if resp.status_code == 200:
                    j = resp.json()
                    data = j.get("data")
                    break
                time.sleep(3 * (attempt + 1))
            except requests.exceptions.RequestException:
                time.sleep(5 * (attempt + 1))
        if data is None or data.get("repository") is None:
            with open(SHA_FAIL_LOG, "a") as f:
                f.write(f"{full_name}\t{numbers}\tfailed\n")
            continue
        repo_data = data["repository"]
        for key, val in repo_data.items():
            if val is None:
                continue
            mc = val.get("mergeCommit")
            sha_results.append({
                "pr_id": pr_id_by_number[val["number"]],
                "full_name": full_name,
                "number": val["number"],
                "base_sha": val.get("baseRefOid"),
                "head_sha": val.get("headRefOid"),
                "merge_sha": mc["oid"] if mc else None,
            })
        pd.DataFrame(sha_results).drop_duplicates(subset=["pr_id"]).to_csv(ALL_SHAS_FILE, index=False)
    print(f"  Done SHAs: {full_name}  (cumulative: {len(sha_results):,})")

print(f"\nTotal SHA records: {len(sha_results):,}")

## Step 11 — Filter to scannable files and group to the PR level

Not every manifest is scannable by OSV-Scanner. For example, `requirements.txt` with un-pinned
versions, or `pyproject.toml` without a lockfile, can't be resolved to concrete package versions.

**Scannable patterns** (OSV-Scanner guaranteed):

- `poetry.lock`, `requirements.txt`
- `package-lock.json`, `yarn.lock`, `pnpm-lock.yaml`
- `go.mod`
- `packages.lock.json` (C# — rare in practice)

We also build `all_pr_groups` — one row per PR with the aggregated set of filenames and the three
SHAs attached. This is the atomic unit for scanning.

In [ ]:
SCANNABLE_PATTERNS = [
    r"poetry\.lock", r"requirements.*\.txt",
    r"package-lock\.json", r"yarn\.lock", r"pnpm-lock\.yaml",
    r"go\.mod",
    r"packages\.lock\.json",
]

# --- AI side ---
ai_rows = ai_df.copy()
ai_rows["is_scannable"] = ai_rows["filename"].apply(
    lambda f: any(re.search(p, f) for p in SCANNABLE_PATTERNS) if isinstance(f, str) else False
)
ai_scannable = ai_rows[ai_rows["is_scannable"]].copy()

# --- Human side ---
human_rows = human_df.copy() if len(human_df) else pd.DataFrame(columns=["pr_id", "full_name", "filename", "ecosystem", "agent"])
if len(human_rows):
    human_rows["is_scannable"] = human_rows["filename"].apply(
        lambda f: any(re.search(p, f) for p in SCANNABLE_PATTERNS) if isinstance(f, str) else False
    )
    human_scannable = human_rows[human_rows["is_scannable"]].copy()
else:
    human_scannable = human_rows

print(f"AI scannable rows:    {len(ai_scannable):,}")
print(f"Human scannable rows: {len(human_scannable):,}")

# Attach SHAs
sha_df = pd.read_csv(ALL_SHAS_FILE) if ALL_SHAS_FILE.exists() else pd.DataFrame()
if len(sha_df):
    sha_cols = sha_df[["pr_id", "base_sha", "head_sha", "merge_sha", "number"]]
    ai_scannable = ai_scannable.merge(sha_cols, on="pr_id", how="left")
    if len(human_scannable):
        human_scannable = human_scannable.merge(sha_cols.drop(columns=["number"]),
                                                on="pr_id", how="left")

# Group to PR level
def _groupby_pr(df, side):
    if len(df) == 0:
        return pd.DataFrame(columns=["pr_id","full_name","base_sha","head_sha","merge_sha",
                                     "ecosystems","filenames","side","agent","number"])
    agg = df.groupby("pr_id").agg(
        full_name=("full_name", "first"),
        base_sha=("base_sha", "first"),
        head_sha=("head_sha", "first"),
        merge_sha=("merge_sha", "first"),
        ecosystems=("ecosystem", lambda x: sorted(set(x))),
        filenames=("filename", lambda x: sorted(set(x))),
        number=("number", "first") if "number" in df.columns else ("full_name", "size"),
    ).reset_index()
    agg["side"] = side
    agg["agent"] = df.groupby("pr_id")["agent"].first().reindex(agg["pr_id"]).values if "agent" in df.columns else side
    return agg

ai_pr_groups = _groupby_pr(ai_scannable, "ai")
human_pr_groups = _groupby_pr(human_scannable, "human") if len(human_scannable) else pd.DataFrame()

all_pr_groups = pd.concat([ai_pr_groups, human_pr_groups], ignore_index=True)
# Drop PRs with no base_sha (can't fetch before-state)
all_pr_groups = all_pr_groups.dropna(subset=["base_sha"]).copy()
print(f"\nTotal PRs ready to scan: {len(all_pr_groups):,}")
print(all_pr_groups["side"].value_counts())

## Step 12 — Fetch manifest content before/after and scan with OSV-Scanner

The core of the whole pipeline. For every PR, for every one of its manifest files:

1. GraphQL-fetch the file at `base_sha` (before the PR)
2. GraphQL-fetch the file at `merge_sha` (after the PR, fallback: `head_sha`)
3. Write both snapshots to temp files
4. Run `osv-scanner scan --format json` on each
5. Parse the JSON into a `{vuln_id: severity}` dict
6. Compute set difference: `introduced = after − before`, `fixed = before − after`

**Two edge cases** worth highlighting:
- GraphQL's blob API returns at most **512 KB of text**. For large lockfiles (very common for
  `pnpm-lock.yaml` in big monorepos) we fall back to the REST `/contents/` endpoint, which returns
  a `download_url` for the full raw file.
- `mergeCommit` is often `null` after a branch deletion — we fall back to `head_sha`.

Output is written row-by-row to `osv_scan_results.csv`. Resumable.

In [ ]:
FETCH_QUERY = """
query($owner: String!, $name: String!, $baseExpr: String!, $afterExpr: String!) {
  repository(owner: $owner, name: $name) {
    baseFile:  object(expression: $baseExpr)  { ... on Blob { text isBinary byteSize } }
    afterFile: object(expression: $afterExpr) { ... on Blob { text isBinary byteSize } }
  }
}
"""

# Resume
done_osv_ids = set()
if OSV_DONE_FILE.exists():
    done_osv_ids = set(int(l) for l in OSV_DONE_FILE.read_text().splitlines() if l.strip())
osv_rows = []
if OSV_RESULTS_FILE.exists():
    osv_rows = pd.read_csv(OSV_RESULTS_FILE).to_dict("records")

print(f"OSV already scanned: {len(done_osv_ids):,}")
print(f"Remaining:           {len(all_pr_groups) - len(done_osv_ids):,}")

_rate_counter = {"n": 0}

for idx, pr in all_pr_groups.iterrows():
    if pr["pr_id"] in done_osv_ids:
        continue
    owner, repo = pr["full_name"].split("/", 1)
    base_sha = pr["base_sha"]
    after_sha = pr["merge_sha"] if isinstance(pr["merge_sha"], str) and pr["merge_sha"] else pr["head_sha"]

    before_vulns = {}
    after_vulns = {}
    scan_had_error = False

    for path in pr["filenames"]:
        # --- Rate-limit check every 50 calls ---
        _rate_counter["n"] += 1
        if _rate_counter["n"] % 50 == 0:
            try:
                rl = requests.get("https://api.github.com/rate_limit", headers=HEADERS, timeout=10).json()
                rem = rl["resources"]["graphql"]["remaining"]
                if rem < 100:
                    wait = max(rl["resources"]["graphql"]["reset"] - time.time(), 0) + 5
                    print(f"  Rate limit low ({rem}); sleeping {wait/60:.1f} min")
                    time.sleep(wait)
            except Exception:
                pass

        # --- Fetch base + after content via GraphQL ---
        base_text = after_text = None
        try:
            resp = requests.post(
                GRAPHQL_URL, headers=HEADERS,
                json={"query": FETCH_QUERY, "variables": {
                    "owner": owner, "name": repo,
                    "baseExpr": f"{base_sha}:{path}",
                    "afterExpr": f"{after_sha}:{path}",
                }},
                timeout=30,
            )
            if resp.status_code == 200:
                d = resp.json().get("data", {}).get("repository", {})
                for key, dest in [("baseFile", "base"), ("afterFile", "after")]:
                    blob = d.get(key)
                    if blob and not blob.get("isBinary"):
                        txt = blob.get("text")
                        # Fall back to REST if truncated (>512KB)
                        if txt is not None and len(txt) < blob.get("byteSize", 0):
                            sha_use = base_sha if key == "baseFile" else after_sha
                            url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}"
                            r2 = requests.get(url, headers=HEADERS, params={"ref": sha_use}, timeout=30)
                            if r2.status_code == 200:
                                j2 = r2.json()
                                dl = j2.get("download_url")
                                if dl:
                                    r3 = requests.get(dl, timeout=30)
                                    if r3.status_code == 200:
                                        txt = r3.text
                        if dest == "base":
                            base_text = txt
                        else:
                            after_text = txt
        except requests.exceptions.RequestException as e:
            with open(OSV_FAIL_LOG, "a") as f:
                f.write(f"{pr['pr_id']}\t{path}\tfetch\t{e}\n")
            scan_had_error = True
            continue

        # --- Scan each snapshot with OSV ---
        for text, target in [(base_text, before_vulns), (after_text, after_vulns)]:
            if not text:
                continue
            tmp_dir = Path(tempfile.mkdtemp(prefix="osv_"))
            try:
                tmp_file = tmp_dir / os.path.basename(path)
                tmp_file.write_text(text, encoding="utf-8", errors="replace")
                result_file = tmp_dir / "result.json"
                r = subprocess.run(
                    [str(OSV_BIN), "scan", "--format", "json", str(tmp_file)],
                    capture_output=True, text=True, timeout=120,
                    encoding="utf-8", errors="replace",
                )
                # OSV writes JSON to stdout
                try:
                    osv_data = json.loads(r.stdout) if r.stdout else {}
                except json.JSONDecodeError:
                    osv_data = {}
                # Extract unique vuln_id -> severity
                for res in osv_data.get("results", []):
                    for pkg in res.get("packages", []):
                        for v in pkg.get("vulnerabilities", []):
                            target[v["id"]] = v.get("database_specific", {}).get("severity")
            except subprocess.TimeoutExpired:
                scan_had_error = True
                with open(OSV_FAIL_LOG, "a") as f:
                    f.write(f"{pr['pr_id']}\t{path}\ttimeout\n")
            except Exception as e:
                scan_had_error = True
                with open(OSV_FAIL_LOG, "a") as f:
                    f.write(f"{pr['pr_id']}\t{path}\tscan\t{e}\n")
            finally:
                shutil.rmtree(tmp_dir, ignore_errors=True)

    introduced = set(after_vulns) - set(before_vulns)
    fixed = set(before_vulns) - set(after_vulns)

    osv_rows.append({
        "pr_id": pr["pr_id"],
        "side": pr["side"],
        "agent": pr["agent"],
        "full_name": pr["full_name"],
        "ecosystems": ";".join(pr["ecosystems"]),
        "num_files_scanned": len(pr["filenames"]),
        "before_count": len(before_vulns),
        "after_count": len(after_vulns),
        "introduced_count": len(introduced),
        "fixed_count": len(fixed),
        "introduced_ids": ";".join(sorted(introduced)),
        "introduced_severities": ";".join((after_vulns[v] or "UNKNOWN") for v in sorted(introduced)),
        "scan_had_error": scan_had_error,
    })

    done_osv_ids.add(pr["pr_id"])
    # Checkpoint every 25 PRs
    if len(osv_rows) % 25 == 0:
        pd.DataFrame(osv_rows).to_csv(OSV_RESULTS_FILE, index=False)
        OSV_DONE_FILE.write_text("\n".join(str(i) for i in done_osv_ids))
        print(f"  OSV checkpoint: {len(done_osv_ids):,}/{len(all_pr_groups):,}")

pd.DataFrame(osv_rows).to_csv(OSV_RESULTS_FILE, index=False)
OSV_DONE_FILE.write_text("\n".join(str(i) for i in done_osv_ids))
print(f"\nOSV scanning complete. {len(osv_rows):,} PRs scanned.")

## Step 13 — Semgrep source-code scan (complementary to OSV)

OSV-Scanner only looks at **dependency manifests**. It can't detect a SQL injection the PR author
wrote in `app.py`, or a hardcoded AWS key. For that we run **Semgrep** over the source files the
PR changed.

For each PR we:

1. Ask GitHub for every file the PR touched (not just manifests).
2. Filter to source files Semgrep understands: `.py`, `.js`, `.ts`, `.jsx`, `.tsx`, `.go`, `.java`,
   `.rb`, `.php`, `.cs`, `.ipynb`.
3. Fetch the base and after version of each file.
4. Run `semgrep scan --config auto --json` on both.
5. Count findings by severity and record before/after/introduced/fixed by `check_id`.

Semgrep is slower than OSV (it loads a ruleset) — expect a few seconds per file. Checkpointed.

In [ ]:
SEMGREP_EXTS = {".py", ".js", ".ts", ".jsx", ".tsx", ".go", ".java", ".rb", ".php", ".cs", ".ipynb"}

FILES_QUERY = """
query($owner: String!, $name: String!, $number: Int!) {
  repository(owner: $owner, name: $name) {
    pullRequest(number: $number) {
      files(first: 100) { nodes { path } }
    }
  }
}
"""

# Resume
done_sg_ids = set()
if SEMGREP_DONE_FILE.exists():
    done_sg_ids = set(int(l) for l in SEMGREP_DONE_FILE.read_text().splitlines() if l.strip())
sg_rows = []
if SEMGREP_RESULTS_FILE.exists():
    sg_rows = pd.read_csv(SEMGREP_RESULTS_FILE).to_dict("records")

print(f"Semgrep already done: {len(done_sg_ids):,}")
print(f"Remaining:            {len(all_pr_groups) - len(done_sg_ids):,}")

for idx, pr in all_pr_groups.iterrows():
    if pr["pr_id"] in done_sg_ids:
        continue
    owner, repo = pr["full_name"].split("/", 1)
    number = int(pr["number"])
    base_sha = pr["base_sha"]
    after_sha = pr["merge_sha"] if isinstance(pr["merge_sha"], str) and pr["merge_sha"] else pr["head_sha"]

    # --- Get the file list for the PR ---
    paths = []
    try:
        resp = requests.post(GRAPHQL_URL, headers=HEADERS,
                             json={"query": FILES_QUERY,
                                   "variables": {"owner": owner, "name": repo, "number": number}},
                             timeout=30)
        if resp.status_code == 200:
            d = resp.json().get("data", {}).get("repository", {}).get("pullRequest")
            if d and d.get("files"):
                paths = [n["path"] for n in d["files"]["nodes"]]
    except requests.exceptions.RequestException:
        pass

    source_paths = [p for p in paths if any(p.lower().endswith(ext) for ext in SEMGREP_EXTS)]

    before_findings = {}  # check_id -> severity
    after_findings = {}

    for path in source_paths[:10]:  # cap to 10 source files per PR to keep runtime reasonable
        # Fetch base + after text (same technique as OSV step)
        base_text = after_text = None
        try:
            resp = requests.post(
                GRAPHQL_URL, headers=HEADERS,
                json={"query": FETCH_QUERY, "variables": {
                    "owner": owner, "name": repo,
                    "baseExpr": f"{base_sha}:{path}",
                    "afterExpr": f"{after_sha}:{path}",
                }},
                timeout=30,
            )
            if resp.status_code == 200:
                d = resp.json().get("data", {}).get("repository", {})
                for key, assign in [("baseFile", "base"), ("afterFile", "after")]:
                    blob = d.get(key)
                    if blob and not blob.get("isBinary"):
                        txt = blob.get("text")
                        if assign == "base":
                            base_text = txt
                        else:
                            after_text = txt
        except requests.exceptions.RequestException:
            continue

        for text, target in [(base_text, before_findings), (after_text, after_findings)]:
            if not text:
                continue
            tmp_dir = Path(tempfile.mkdtemp(prefix="sg_"))
            try:
                tmp_file = tmp_dir / os.path.basename(path)
                tmp_file.write_text(text, encoding="utf-8", errors="replace")
                r = subprocess.run(
                    ["semgrep", "scan", "--config", "auto", "--json", "--quiet",
                     "--timeout", "30", str(tmp_file)],
                    capture_output=True, text=True, timeout=120, shell=True,
                    encoding="utf-8", errors="replace",
                )
                try:
                    sg_data = json.loads(r.stdout) if r.stdout else {}
                except json.JSONDecodeError:
                    sg_data = {}
                for finding in sg_data.get("results", []):
                    cid = finding.get("check_id", "unknown")
                    sev = finding.get("extra", {}).get("severity", "INFO")
                    target[f"{cid}@{os.path.basename(path)}"] = sev
            except subprocess.TimeoutExpired:
                pass
            except Exception as e:
                with open(SEMGREP_FAIL_LOG, "a") as f:
                    f.write(f"{pr['pr_id']}\t{path}\t{e}\n")
            finally:
                shutil.rmtree(tmp_dir, ignore_errors=True)

    introduced = set(after_findings) - set(before_findings)
    fixed = set(before_findings) - set(after_findings)

    sg_rows.append({
        "pr_id": pr["pr_id"],
        "side": pr["side"],
        "agent": pr["agent"],
        "full_name": pr["full_name"],
        "num_source_files_scanned": len(source_paths[:10]),
        "sg_before_count": len(before_findings),
        "sg_after_count": len(after_findings),
        "sg_introduced_count": len(introduced),
        "sg_fixed_count": len(fixed),
        "sg_introduced_severities": ";".join(after_findings[v] for v in sorted(introduced)),
    })

    done_sg_ids.add(pr["pr_id"])
    if len(sg_rows) % 10 == 0:
        pd.DataFrame(sg_rows).to_csv(SEMGREP_RESULTS_FILE, index=False)
        SEMGREP_DONE_FILE.write_text("\n".join(str(i) for i in done_sg_ids))
        print(f"  Semgrep checkpoint: {len(done_sg_ids):,}/{len(all_pr_groups):,}")

pd.DataFrame(sg_rows).to_csv(SEMGREP_RESULTS_FILE, index=False)
SEMGREP_DONE_FILE.write_text("\n".join(str(i) for i in done_sg_ids))
print(f"\nSemgrep scanning complete. {len(sg_rows):,} PRs scanned.")

## Step 14 — Fetch review engagement + merge-time metadata

For the exploratory part of the paper we need per-PR:
- `state` (MERGED / CLOSED / OPEN) — we'll filter to `MERGED` only
- `created_at` / `merged_at` → `hours_to_merge`
- `review_count` — number of PR reviews posted
- `comment_count` — number of top-level issue comments on the PR

Batched 50 PRs per repo via GraphQL aliases, same pattern as the SHA fetch.

In [ ]:
REVIEW_BATCH = 50

done_review_ids = set()
review_rows = []
if REVIEW_META_FILE.exists():
    rvdf_prev = pd.read_csv(REVIEW_META_FILE)
    review_rows = rvdf_prev.to_dict("records")
    done_review_ids = set(rvdf_prev["pr_id"])

print(f"Review metadata done: {len(done_review_ids):,}")
remaining = all_pr_groups[~all_pr_groups["pr_id"].isin(done_review_ids)]
print(f"Remaining:            {len(remaining):,}")

for full_name, group in remaining.groupby("full_name"):
    owner, repo = full_name.split("/", 1)
    rows = group[["pr_id", "number"]].to_dict("records")
    for i in range(0, len(rows), REVIEW_BATCH):
        batch = rows[i:i + REVIEW_BATCH]
        numbers = [int(p["number"]) for p in batch]
        id_by_num = {int(p["number"]): p["pr_id"] for p in batch}
        aliases = "\n".join(
            f'pr{j}: pullRequest(number: {n}) {{ number createdAt mergedAt state '
            f'reviews {{ totalCount }} comments {{ totalCount }} }}'
            for j, n in enumerate(numbers)
        )
        query = f'query {{ repository(owner: "{owner}", name: "{repo}") {{ {aliases} }} }}'
        for attempt in range(5):
            try:
                resp = requests.post(GRAPHQL_URL, headers=HEADERS,
                                     json={"query": query}, timeout=30)
                if resp.status_code == 200:
                    j = resp.json()
                    d = j.get("data", {}).get("repository")
                    if d:
                        for key, val in d.items():
                            if val is None:
                                continue
                            review_rows.append({
                                "pr_id": id_by_num[val["number"]],
                                "full_name": full_name,
                                "created_at": val.get("createdAt"),
                                "merged_at": val.get("mergedAt"),
                                "state": val.get("state"),
                                "review_count": val["reviews"]["totalCount"] if val.get("reviews") else None,
                                "comment_count": val["comments"]["totalCount"] if val.get("comments") else None,
                            })
                    break
                time.sleep(3 * (attempt + 1))
            except requests.exceptions.RequestException:
                time.sleep(5 * (attempt + 1))
        pd.DataFrame(review_rows).drop_duplicates(subset=["pr_id"]).to_csv(REVIEW_META_FILE, index=False)
    print(f"  Reviews done: {full_name}  (cumulative: {len(review_rows):,})")

print(f"\nTotal review metadata rows: {len(review_rows):,}")

## Step 15 — Assemble the final analysis-ready dataset

Join the three result tables (OSV, Semgrep, review-metadata) on `pr_id`, filter to **merged PRs
only**, compute `hours_to_merge`, and save the final CSV.

This is the file equivalent to `paper2_final_analysis_dataset.csv` in the reference repo —
analysis-ready input for the RQ1 (proportion test) and RQ2 (severity comparison) scripts.

In [ ]:
osv_df = pd.read_csv(OSV_RESULTS_FILE)
sg_df = pd.read_csv(SEMGREP_RESULTS_FILE)
rv_df = pd.read_csv(REVIEW_META_FILE).drop_duplicates(subset=["pr_id"])

print(f"osv_df:    {len(osv_df):,} rows")
print(f"sg_df:     {len(sg_df):,} rows")
print(f"review_df: {len(rv_df):,} rows")

# Merge OSV + Semgrep (one-to-one on pr_id)
final = osv_df.merge(
    sg_df[["pr_id", "num_source_files_scanned", "sg_before_count", "sg_after_count",
           "sg_introduced_count", "sg_fixed_count", "sg_introduced_severities"]],
    on="pr_id", how="left",
)

# Attach review metadata
final = final.merge(
    rv_df[["pr_id", "created_at", "merged_at", "state", "review_count", "comment_count"]],
    on="pr_id", how="left",
)

# Filter: merged only
before_n = len(final)
final = final[final["state"] == "MERGED"].copy()
print(f"\nFiltered to merged-only: {before_n:,} -> {len(final):,}")

# Derived: hours to merge
final["created_at"] = pd.to_datetime(final["created_at"], errors="coerce")
final["merged_at"] = pd.to_datetime(final["merged_at"], errors="coerce")
final["hours_to_merge"] = (final["merged_at"] - final["created_at"]).dt.total_seconds() / 3600

# Overall summary
print("\n--- Side breakdown ---")
print(final["side"].value_counts())
print("\n--- Agent breakdown (AI side) ---")
print(final[final["side"] == "ai"]["agent"].value_counts())
print("\n--- Introduced vuln rate (OSV) ---")
print(final.assign(has_vuln=final["introduced_count"] > 0)
      .groupby("side")["has_vuln"].mean().mul(100).round(2).astype(str) + " %")
print("\n--- Introduced finding rate (Semgrep) ---")
print(final.assign(has_sg=final["sg_introduced_count"] > 0)
      .groupby("side")["has_sg"].mean().mul(100).round(2).astype(str) + " %")

final.to_csv(FINAL_FILE, index=False)
print(f"\nFinal dataset saved: {FINAL_FILE}  ({len(final):,} rows)")
print(f"Columns: {list(final.columns)}")

## Step 16 — Quick-look RQ1: do AI PRs introduce vulns at a different rate?

A fast-feedback 2×2 proportion test (chi-square + Fisher's exact + odds ratio) mirroring
`analysis_scripts/rq1_proportion_test.py` from the reference repo. This lets you confirm the
end-to-end pipeline produced sane data before you run the full statistical scripts.

In [ ]:
from scipy import stats
import math

tbl = pd.crosstab(
    final["side"],
    (final["introduced_count"] > 0).map({True: "introduced", False: "clean"}),
)
print(tbl)

if {"ai", "human"}.issubset(tbl.index) and "introduced" in tbl.columns:
    a = int(tbl.loc["ai", "introduced"])
    b = int(tbl.loc["ai", "clean"]) if "clean" in tbl.columns else 0
    c = int(tbl.loc["human", "introduced"])
    d = int(tbl.loc["human", "clean"]) if "clean" in tbl.columns else 0
    chi2, p_chi, _, _ = stats.chi2_contingency([[a, b], [c, d]], correction=True)
    odds, p_fisher = stats.fisher_exact([[a, b], [c, d]])
    try:
        se = math.sqrt(1/a + 1/b + 1/c + 1/d)
        lo = math.exp(math.log(odds) - 1.96 * se)
        hi = math.exp(math.log(odds) + 1.96 * se)
    except (ZeroDivisionError, ValueError):
        lo = hi = float("nan")
    print(f"\nChi-square p = {p_chi:.3g}")
    print(f"Fisher exact p = {p_fisher:.3g}")
    print(f"Odds ratio (AI vs human) = {odds:.2f}   [95% CI {lo:.2f}, {hi:.2f}]")
    print(f"AI intro rate:    {a/(a+b)*100:.2f}%  ({a}/{a+b})")
    print(f"Human intro rate: {c/(c+d)*100:.2f}%  ({c}/{c+d})")
else:
    print("\nNot enough data for a 2x2 test yet.")